# Fine-tuned evaluation

GPU T4. Base model in fp16 with the merged LoRA adapter, on the same test sets as the zero-shot run. Also measures latency and peak memory at batch size 1.

Secret required: `HF_TOKEN`.

In [ ]:
import json
import os
import subprocess

REPO = "https://github.com/N-Mehdi/receipt-vlm.git"
DATASET = "N-Mehdi/french-invoices-synth"
ADAPTER = "N-Mehdi/qwen3-vl-2b-receipts-lora"
OUT = "/kaggle/working/outputs"


def run(cmd):
    """Run a shell command, stream its output and fail the notebook on error."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError("command failed: " + cmd)


run("git clone --depth 1 " + REPO + " /tmp/receipt-vlm")
os.chdir("/tmp/receipt-vlm")
run("git log -1 --format=%H")
run("pip install -q -r requirements.txt && pip install -q --no-deps -e .")
run(
    'python -c \'import torch, transformers; print("torch", torch.__version__,'
    ' "cuda", torch.version.cuda, "transformers", transformers.__version__)\''
)

In [ ]:
from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()
os.environ["HF_TOKEN"] = client.get_secret("HF_TOKEN")

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
run("nvidia-smi --query-gpu=name,memory.total --format=csv")

In [ ]:
PREFIX = "finetuned"
EXTRA = " --adapter " + ADAPTER + " --latency-docs 20"
run(f"python scripts/evaluate.py --dataset cord{EXTRA} --out {OUT}/{PREFIX}_cord.json")
run(
    f"python scripts/evaluate.py --dataset invoice --invoice-source {DATASET}{EXTRA}"
    f" --out {OUT}/{PREFIX}_invoice.json"
)

In [ ]:
NAMES = ["finetuned_cord", "finetuned_invoice"]
for name in NAMES:
    report = json.load(open(OUT + "/" + name + ".json"))
    for group in ["all", "seen", "unseen"]:
        if group in report:
            r = report[group]
            print(name, group, "n", r["n"], "valid json", round(r["valid_json"], 3))
            print("  exact match", round(r["exact_match"], 3), r["exact_match_ci"])
            print("  f1", round(r["f1"], 3), r["f1_ci"], "truncated", r["truncated_rate"])
    if "latency" in report:
        print(name, "latency", report["latency"])
    print(name, "seconds", round(report["seconds"]))